In [2]:
import numpy as np 
import pandas as pd
df=pd.read_csv(r"C:\Users\spandana.s\OneDrive\Desktop\MUFG\manufacturing_dataset_1000_samples.csv")
# Convert Timestamp to datetime
df["Timestamp"] = pd.to_datetime(df["Timestamp"])

# Extract useful time-based features
df["Hour"] = df["Timestamp"].dt.hour
df["Day"] = df["Timestamp"].dt.day
df["Month"] = df["Timestamp"].dt.month
df["DayOfWeek_Num"] = df["Timestamp"].dt.dayofweek  # Monday=0, Sunday=6

# Drop the original Timestamp column
df = df.drop(columns=["Timestamp"])

# Preview updated dataset
df.head()


,Injection_Temperature,Injection_Pressure,Cycle_Time,Cooling_Time,Material_Viscosity,Ambient_Temperature,Machine_Age,Operator_Experience,Maintenance_Hours,Shift,...,Day_of_Week,Temperature_Pressure_Ratio,Total_Cycle_Time,Efficiency_Score,Machine_Utilization,Parts_Per_Hour,Hour,Day,Month,DayOfWeek_Num
0,221.0,136.0,28.7,13.6,375.5,28.0,3.8,11.2,64,Evening,...,Thursday,1.625,42.3,0.063,0.510,36.5,0,1,1,6
1,213.3,128.9,34.5,14.0,215.8,22.6,6.8,6.3,58,Night,...,Wednesday,1.655,48.5,0.037,0.389,29.9,1,1,1,6
2,222.8,115.9,19.9,9.5,307.0,25.3,4.2,9.6,47,Day,...,Monday,1.922,29.4,0.061,0.551,56.9,2,1,1,6
3,233.3,105.3,39.2,13.1,137.8,26.0,9.2,8.6,49,Evening,...,Saturday,2.215,52.3,0.054,0.293,31.0,3,1,1,6
4,212.2,125.5,45.0,9.9,298.2,23.6,6.2,23.0,49,Night,...,Monday,1.691,54.9,0.145,0.443,15.0,4,1,1,6


In [3]:
# Numerical columns -> fill with median
num_cols = ["Material_Viscosity", "Ambient_Temperature", "Operator_Experience"]
for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Categorical columns -> fill with mode
cat_cols = df.select_dtypes(include=["object"]).columns
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Check if missing values are handled
print(df.isnull().sum())


Injection_Temperature         0
Injection_Pressure            0
Cycle_Time                    0
Cooling_Time                  0
Material_Viscosity            0
Ambient_Temperature           0
Machine_Age                   0
Operator_Experience           0
Maintenance_Hours             0
Shift                         0
Machine_Type                  0
Material_Grade                0
Day_of_Week                   0
Temperature_Pressure_Ratio    0
Total_Cycle_Time              0
Efficiency_Score              0
Machine_Utilization           0
Parts_Per_Hour                0
Hour                          0
Day                           0
Month                         0
DayOfWeek_Num                 0
dtype: int64


C:\Users\spandana.s\AppData\Local\Temp\ipykernel_7884\2273481048.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
C:\Users\spandana.s\AppData\Local\Temp\ipykernel_7884\2273481048.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

F

In [4]:
categorical_cols = ["Shift", "Machine_Type", "Material_Grade", "Day_of_Week"]

# Apply One-Hot Encoding (drop_first avoids dummy variable trap)
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Show new column names after encoding
print(df_encoded.columns)
print(df_encoded.head())


Index(['Injection_Temperature', 'Injection_Pressure', 'Cycle_Time',
       'Cooling_Time', 'Material_Viscosity', 'Ambient_Temperature',
       'Machine_Age', 'Operator_Experience', 'Maintenance_Hours',
       'Temperature_Pressure_Ratio', 'Total_Cycle_Time', 'Efficiency_Score',
       'Machine_Utilization', 'Parts_Per_Hour', 'Hour', 'Day', 'Month',
       'DayOfWeek_Num', 'Shift_Evening', 'Shift_Night', 'Machine_Type_Type_B',
       'Machine_Type_Type_C', 'Material_Grade_Premium',
       'Material_Grade_Standard', 'Day_of_Week_Monday', 'Day_of_Week_Saturday',
       'Day_of_Week_Sunday', 'Day_of_Week_Thursday', 'Day_of_Week_Tuesday',
       'Day_of_Week_Wednesday'],
      dtype='object')
   Injection_Temperature  Injection_Pressure  Cycle_Time  Cooling_Time  \
0                  221.0               136.0        28.7          13.6   
1                  213.3               128.9        34.5          14.0   
2                  222.8               115.9        19.9           9.5   
3      

In [5]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler


# Select numerical features (exclude categorical)
num_cols = [
    "Injection_Temperature", "Injection_Pressure", "Cycle_Time",
    "Cooling_Time", "Material_Viscosity", "Ambient_Temperature",
    "Machine_Age", "Operator_Experience", "Maintenance_Hours",
    "Temperature_Pressure_Ratio", "Total_Cycle_Time",
    "Efficiency_Score", "Machine_Utilization", "Parts_Per_Hour"
]

# Standardization (mean=0, std=1)
scaler_standard = StandardScaler()
df_standard_scaled = df.copy()
df_standard_scaled[num_cols] = scaler_standard.fit_transform(df[num_cols])

# Min-Max Scaling (0–1 range)
scaler_minmax = MinMaxScaler()
df_minmax_scaled = df.copy()
df_minmax_scaled[num_cols] = scaler_minmax.fit_transform(df[num_cols])

# Show scaled data (Standard Scaler)
print("Standard Scaler (first 5 rows):")
print(df_standard_scaled[num_cols].head())

# Show scaled data (MinMax Scaler)
print("\nMinMax Scaler (first 5 rows):")
print(df_minmax_scaled[num_cols].head())


Standard Scaler (first 5 rows):
   Injection_Temperature  Injection_Pressure  Cycle_Time  Cooling_Time  \
0               0.474090            1.359149   -0.856562      0.728050   
1              -0.168139            0.874835   -0.161894      0.901726   
2               0.624221           -0.011937   -1.910541     -1.052130   
3               1.499987           -0.734998    0.401027      0.510955   
4              -0.259886            0.642910    1.095695     -0.878454   

   Material_Viscosity  Ambient_Temperature  Machine_Age  Operator_Experience  \
0            1.708997             1.843582    -1.040282            -0.708301   
1           -0.491176            -0.124033    -0.270824            -0.887005   
2            0.765278             0.859775    -0.937688            -0.766653   
3           -1.565775             1.114836     0.344743            -0.803123   
4            0.644042             0.240341    -0.424715            -0.277951   

   Maintenance_Hours  Temperature_Pressure

In [6]:
# Target variable
y = df["Parts_Per_Hour"]

# Features = all other columns except target
X = df.drop(columns=["Parts_Per_Hour"])

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("Features:\n", X.head())
print("Target:\n", y.head())


Feature shape: (1000, 21)
Target shape: (1000,)
Features:
    Injection_Temperature  Injection_Pressure  Cycle_Time  Cooling_Time  \
0                  221.0               136.0        28.7          13.6   
1                  213.3               128.9        34.5          14.0   
2                  222.8               115.9        19.9           9.5   
3                  233.3               105.3        39.2          13.1   
4                  212.2               125.5        45.0           9.9   

   Material_Viscosity  Ambient_Temperature  Machine_Age  Operator_Experience  \
0               375.5                 28.0          3.8                 11.2   
1               215.8                 22.6          6.8                  6.3   
2               307.0                 25.3          4.2                  9.6   
3               137.8                 26.0          9.2                  8.6   
4               298.2                 23.6          6.2                 23.0   

   Maintenance_

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
# Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)

Training set size: (800, 21)
Test set size: (200, 21)


In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    recall_score, precision_score, f1_score, accuracy_score,
    mean_squared_error, classification_report
)


target = "Efficiency_Score"
X = df.drop(columns=[target])
y = df[target]

# One-hot encode categorical features
X = pd.get_dummies(X, drop_first=True)

# ================================
# Regression Evaluation
# ================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

reg_model = LinearRegression()
reg_model.fit(X_train, y_train)
y_pred_reg = reg_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred_reg)
rmse = np.sqrt(mse)

# ================================
# Classification Evaluation
# ================================
threshold = y.median()
y_class = (y >= threshold).astype(int)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_class, test_size=0.2, random_state=42
)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_c, y_train_c)
y_pred_class = clf.predict(X_test_c)

recall = recall_score(y_test_c, y_pred_class)
precision = precision_score(y_test_c, y_pred_class)
f1 = f1_score(y_test_c, y_pred_class)
accuracy = accuracy_score(y_test_c, y_pred_class)

# ================================
# Model Evaluation Report
# ================================
print("\n=== Model Evaluation Report ===")
print("Primary Metrics:")
print(f"- Recall (Sensitivity): {recall:.4f}")
print(f"- Precision           : {precision:.4f}")
print(f"- F1-score            : {f1:.4f}")
print(f"- MSE                 : {mse:.4f}")
print(f"- RMSE                : {rmse:.4f}")

print("\nSecondary Metric:")
print(f"- Accuracy            : {accuracy:.4f}")

print("\nDetailed Classification Report:")
print(classification_report(y_test_c, y_pred_class))

print("\nRecommendations:")
print("- Use regression metrics (MSE, RMSE) if predicting continuous Efficiency_Score.")
print("- Use classification metrics (Recall, Precision, F1) if categorizing efficiency levels.")
print("- Improve model performance with feature engineering and advanced algorithms (e.g., Random Forest, XGBoost).")
print("- Address missing values, analyze correlations, and check multicollinearity during EDA.")
print("- Document assumptions, preprocessing steps, and justify model choice for clarity and professionalism.")



=== Model Evaluation Report ===
Primary Metrics:
- Recall (Sensitivity): 0.9810
- Precision           : 0.9626
- F1-score            : 0.9717
- MSE                 : 0.0013
- RMSE                : 0.0365

Secondary Metric:
- Accuracy            : 0.9700

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.96      0.97        95
           1       0.96      0.98      0.97       105

    accuracy                           0.97       200
   macro avg       0.97      0.97      0.97       200
weighted avg       0.97      0.97      0.97       200


Recommendations:
- Use regression metrics (MSE, RMSE) if predicting continuous Efficiency_Score.
- Use classification metrics (Recall, Precision, F1) if categorizing efficiency levels.
- Improve model performance with feature engineering and advanced algorithms (e.g., Random Forest, XGBoost).
- Address missing values, analyze correlations, and check multicollinearity during EDA.
- 

C:\Users\spandana.s\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
